# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [1]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 17:18:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Versión de Spark: 4.1.1


In [2]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)


Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [3]:
# TODO
ruta_clientes = "/opt/spark-apps/datos/clientes.csv"
ruta_pedidos = "/opt/spark-apps/datos/pedidos.csv"

df_clientes = spark.read.csv(
    ruta_clientes,
    header=True,
    inferSchema=True,
    sep=";"
)

df_pedidos = spark.read.csv(
    ruta_pedidos,
    header=True,
    inferSchema=True,
    sep=";"
)

# Filas de ejemplo.
print("==== FILAS ====") 
print("CLIENTES ") 
df_clientes.show()
print("\nPEDIDOS ") 
df_pedidos.show()

# Esquema
print("\n==== ESQUEMAS ====") 
df_clientes.printSchema()
df_pedidos.printSchema()

==== FILAS ====
CLIENTES 
+----------+------+----------+--------+
|id_cliente|nombre|    ciudad|segmento|
+----------+------+----------+--------+
|         1|   Ana|  Sevilla |Estandar|
|         2|  Luis|    Bilbao| Premium|
|         3| Marta| Alicante |Estandar|
|         4| Pablo|   Madrid | Premium|
|         5| Lucia|    Bilbao| Premium|
|         6|Carlos| Alicante |Estandar|
|         7| Elena|    Bilbao|Estandar|
|         8|Javier|    Madrid| Premium|
|         9|  Sara|    Murcia| Premium|
|        10| David|    Bilbao|Estandar|
|        11|Raquel|  Sevilla | Premium|
|        12|Miguel|  Zaragoza|Estandar|
|        13| Irene|    Madrid| Premium|
|        14|Sergio|   Murcia |Estandar|
|        15| Paula|   Granada|Estandar|
|        16| Diego|  Granada | Premium|
|        17|  Alba|    Madrid|Estandar|
|        18| Jorge|   Sevilla| Premium|
|        19| Clara|    Murcia|Estandar|
|        20|Adrian|  Valencia| Premium|
+----------+------+----------+--------+
only showing t

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [4]:
# ELIMINAR FILAS DUPLICADAS
print("ELIMINAR FILAS DUPLICAS")

# 1. Número de filas que tiene el DataFrame original de clientes
print("\n=== Número de filas de clientes ===")
print("Filas en clientes:", df_clientes.count())

# 2. Mostramos una muestra de los datos para inspección visual
print("\n=== Tabla clientes ===")
df_clientes.show(10, truncate=False)

# 3. Guardamos el número total de filas 
total_clientes = df_clientes.count()

# 4. Calculamos cuántas filas quedarían al eliminar duplicados 
clientes_sin_duplicados = df_clientes.dropDuplicates().count()

# 5. Mostramos el total de filas y cuántos duplicados se han detectado
print("\n=== Comprobación de duplicados ===")
print("Total filas clientes:", total_clientes)
print("Duplicados detectados:", total_clientes - clientes_sin_duplicados)

# 6. Creamos un nuevo DataFrame sin filas duplicadas
df_clientes_limpio = df_clientes.dropDuplicates()

# 7. Comprobamos el resultado tras la limpieza de duplicados
print("\n=== Resultado tras eliminar duplicados ===")
print("Filas en clientes original:", df_clientes.count())
print("Filas en clientes sin duplicados:", df_clientes_limpio.count())


ELIMINAR FILAS DUPLICAS

=== Número de filas de clientes ===
Filas en clientes: 43

=== Tabla clientes ===
+----------+------+----------+--------+
|id_cliente|nombre|ciudad    |segmento|
+----------+------+----------+--------+
|1         |Ana   | Sevilla  |Estandar|
|2         |Luis  |Bilbao    |Premium |
|3         |Marta | Alicante |Estandar|
|4         |Pablo | Madrid   |Premium |
|5         |Lucia |Bilbao    |Premium |
|6         |Carlos| Alicante |Estandar|
|7         |Elena |Bilbao    |Estandar|
|8         |Javier|Madrid    |Premium |
|9         |Sara  |Murcia    |Premium |
|10        |David |Bilbao    |Estandar|
+----------+------+----------+--------+
only showing top 10 rows

=== Comprobación de duplicados ===
Total filas clientes: 43
Duplicados detectados: 3

=== Resultado tras eliminar duplicados ===
Filas en clientes original: 43
Filas en clientes sin duplicados: 40


In [6]:
# ELIMINAR NULOS
print("\nELIMINAR VALORES NULOS")

from pyspark.sql.functions import col, sum, when

# 1. Contamos cuántos valores nulos hay en cada columna de clientes
print("\n=== Valores nulos en clientes ===")
df_clientes_limpio.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_clientes_limpio.columns
]).show()

# 2. Contamos cuántos valores nulos hay en cada columna de pedidos
print("=== Valores nulos en pedidos ===")
df_pedidos.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_pedidos.columns
]).show()

# 3. Mostramos cuántas filas tiene pedidos antes de limpiar nulos
print("=== Pedidos antes de limpiar nulos ===")
print("Filas en pedidos:", df_pedidos.count())

# 4. Eliminamos filas con nulos en las columnas críticas para el análisis
df_pedidos_limpio = df_pedidos.dropna(subset=["producto", "cantidad"])

# 5. Mostramos cuántas filas quedan después de limpiar
print("\n=== Pedidos después de limpiar nulos ===")
print("Filas en pedidos sin nulos:", df_pedidos_limpio.count())
print("Filas eliminadas:", df_pedidos.count() - df_pedidos_limpio.count())


ELIMINAR VALORES NULOS

=== Valores nulos en clientes ===
+----------+------+------+--------+
|id_cliente|nombre|ciudad|segmento|
+----------+------+------+--------+
|         0|     0|     0|       0|
+----------+------+------+--------+

=== Valores nulos en pedidos ===
+---------+----------+-----+--------+--------+---------------+
|id_pedido|id_cliente|fecha|producto|cantidad|precio_unitario|
+---------+----------+-----+--------+--------+---------------+
|        0|         0|    0|       2|       3|              0|
+---------+----------+-----+--------+--------+---------------+

=== Pedidos antes de limpiar nulos ===
Filas en pedidos: 120

=== Pedidos después de limpiar nulos ===
Filas en pedidos sin nulos: 115
Filas eliminadas: 5


In [7]:
# ELIMINAR ESPACIOS
print("\nELIMINAR ESPACIOS")

from pyspark.sql.functions import trim, col

# 1. Limpiamos espacios al inicio y al final en las columnas de texto
df_clientes_limpio = df_clientes_limpio.withColumn("nombre", trim(col("nombre"))) \
                                       .withColumn("ciudad", trim(col("ciudad"))) \
                                       .withColumn("segmento", trim(col("segmento")))

# 2. De pedido
df_pedidos_limpio = df_pedidos_limpio.withColumn("producto", trim(col("producto")))

# 3. Mostramos una muestra después de la limpieza
print("\n=== Muestra después de limpiar espacios ===")
df_clientes_limpio.show(10, truncate=False)
df_pedidos_limpio.show(10, truncate=False)



ELIMINAR ESPACIOS

=== Muestra después de limpiar espacios ===
+----------+--------+--------+--------+
|id_cliente|nombre  |ciudad  |segmento|
+----------+--------+--------+--------+
|18        |Jorge   |Sevilla |Premium |
|40        |Tomas   |Valencia|Estandar|
|5         |Lucia   |Bilbao  |Premium |
|6         |Carlos  |Alicante|Estandar|
|14        |Sergio  |Murcia  |Estandar|
|8         |Javier  |Madrid  |Premium |
|17        |Alba    |Madrid  |Estandar|
|4         |Pablo   |Madrid  |Premium |
|2         |Luis    |Bilbao  |Premium |
|33        |Patricia|Alicante|Premium |
+----------+--------+--------+--------+
only showing top 10 rows
+---------+----------+----------+-----------+--------+---------------+
|id_pedido|id_cliente|fecha     |producto   |cantidad|precio_unitario|
+---------+----------+----------+-----------+--------+---------------+
|1001     |12        |2025-03-05|Ratón      |6.0     |16             |
|1002     |32        |2025-02-02|Ratón      |3.0     |19           

In [ ]:
# RESULTADO FINAL
print("\n=== TABLAS LIMPIADAS ===")
df_clientes_limpio.show(10, truncate=False)
df_pedidos_limpio.show(10, truncate=False)

## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [ ]:
# TODO

## 4. Haz un join entre clientes y pedidos.

In [ ]:
# TODO

## 5. Filtra ventas Premium con importe >= 100.

In [ ]:
# TODO

## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [ ]:
# TODO

## 7. Calcula agregaciones por ciudad y segmento.

In [ ]:
# TODO

## 8. Crea una vista temporal y haz una consulta SQL.

In [ ]:
# TODO

## 9. Usa `sample()` y `randomSplit()`.

In [ ]:
# TODO

## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [ ]:
# TODO

## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

## 12. Cierra la sesión de Spark.

In [ ]:
spark.stop()